# Tool Calling with Function Dispatch

Tool calling lets the LLM decide when to invoke your code, execute it, and resume the conversation with the result. The model never runs the function itself — it signals intent via a structured response, and you run the code and feed the result back in.

In [11]:
import os
import json
import sqlite3
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [12]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL = "gpt-4o-mini"

### SQLite Data Source

Tools should read from real data, not hardcoded dicts.  
SQLite gives us a persistent store without any server setup.

In [13]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
    )
    conn.commit()


def _seed_prices():
    data = {
        "london": 799,
        "paris": 899,
        "tokyo": 1420,
        "berlin": 499,
        "sydney": 2999,
        "new york": 349,
    }
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        for city, price in data.items():
            cursor.execute(
                "INSERT INTO prices (city, price) VALUES (?, ?)"
                " ON CONFLICT(city) DO UPDATE SET price = ?",
                (city, price, price),
            )
        conn.commit()


_seed_prices()
print("DB seeded")

DB seeded


### Python Functions the Model Can Request

These are plain Python functions but the model cannot call them directly; it signals intent and we run them.

In [14]:
def get_ticket_price(city: str) -> str:
    """Look up a return ticket price from the SQLite DB."""
    print(f"TOOL CALLED: get_ticket_price({city!r})", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(),))
        row = cursor.fetchone()
    if row:
        return f"A return ticket to {city.title()} costs ${row[0]:.0f}."
    return f"No price data available for '{city}'."


def set_ticket_price(city: str, price: float) -> str:
    """Upsert a ticket price in the SQLite DB."""
    print(f"TOOL CALLED: set_ticket_price({city!r}, {price})", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?)"
            " ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price),
        )
        conn.commit()
    return f"Price for {city.title()} updated to ${price:.0f}."


# Verify both functions work against the DB
print(get_ticket_price("Tokyo"))
print(set_ticket_price("Dubai", 1150))
print(get_ticket_price("Dubai"))

TOOL CALLED: get_ticket_price('Tokyo')
A return ticket to Tokyo costs $1420.
TOOL CALLED: set_ticket_price('Dubai', 1150)
Price for Dubai updated to $1150.
TOOL CALLED: get_ticket_price('Dubai')
A return ticket to Dubai costs $1150.


### Defining Tools as JSON Schema

Each tool is a dict with `type: "function"` and a nested `function` object.  
The `description` guides when the model chooses this tool.  
The `parameters` schema tells it what arguments to pass.

```
tool
├── type             "function"
└── function
    ├── name         must match the Python function name
    ├── description  plain English — what it does and when to use it
    └── parameters   JSON Schema object describing the arguments
```

In [15]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Look up the price of a return airline ticket to a destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The destination city (e.g. 'London', 'Tokyo').",
                    }
                },
                "required": ["city"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_ticket_price",
            "description": "Update or create the ticket price for a destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The destination city.",
                    },
                    "price": {
                        "type": "number",
                        "description": "The new return ticket price in USD.",
                    },
                },
                "required": ["city", "price"],
                "additionalProperties": False,
            },
        },
    },
]

print(f"{len(tools)} tools registered")

2 tools registered


### Function Dispatch

A registry maps tool names to Python callables.  
`dispatch_tool` parses the model's JSON arguments and routes to the right function.

In [16]:
TOOL_REGISTRY = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price,
}


def dispatch_tool(tool_call) -> dict:
    """
    Execute a single tool call from the model's response.
    Returns a role:'tool' message ready to append to the messages list.
    """
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)  # always valid JSON

    fn = TOOL_REGISTRY.get(name)
    if fn is None:
        result = f"Unknown tool: {name}"
    else:
        result = fn(**args)

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result,
    }

### The Tool-Calling Loop

```
while finish_reason == "tool_calls":
    1. append assistant message (with tool_calls)
    2. run each requested tool → append role:"tool" messages
    3. call the API again with the updated messages list
```

The loop exits when `finish_reason == "stop"` — the model is satisfied with the tool results and produces a final answer.

In [17]:
SYSTEM = (
    "You are a helpful assistant for FlightAI, an airline booking service. "
    "Give short, accurate answers. "
    "Use the available tools to look up or update ticket prices when needed."
)


def chat(message: str, history: list[dict]) -> str:
    """
    Gradio ChatInterface callback.
    Loops until finish_reason is 'stop', executing any tool calls along the way.
    """
    messages = [{"role": "system", "content": SYSTEM}] + history + [
        {"role": "user", "content": message}
    ]

    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_results = [dispatch_tool(tc) for tc in assistant_msg.tool_calls]

        messages.append(assistant_msg)     # assistant turn with tool_calls
        messages.extend(tool_results)      # one role:"tool" per call

        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    return response.choices[0].message.content

### Verify: Tool Called Only When Needed

Ask a price question → tool fires.  
Ask something unrelated → no tool call, model answers directly.

In [18]:
print("--- Price question (tool should fire) ---")
reply = chat("How much is a flight to Tokyo?", [])
print("Reply:", reply)

print()
print("--- Unrelated question (no tool) ---")
reply = chat("What time does FlightAI's customer service open?", [])
print("Reply:", reply)

--- Price question (tool should fire) ---
TOOL CALLED: get_ticket_price('Tokyo')
Reply: A return ticket to Tokyo costs $1420.

--- Unrelated question (no tool) ---
Reply: FlightAI's customer service typically operates from 8 AM to 8 PM local time, but it's best to check their official website for specific hours and any potential changes.


### Gradio Chat UI

Tool calls are transparent to the user — they see only the final answer.  
Watch the console for `TOOL CALLED:` lines to see the dispatch happening.

In [19]:
demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="FlightAI Assistant",
    description=(
        "Ask about ticket prices or request price updates. "
        "Watch the console to see tool calls fire in real time."
    ),
    examples=[
        "How much does a flight to London cost?",
        "What's the price difference between Paris and Berlin?",
        "Set the price for Rome to $650.",
        "Compare prices for Tokyo and Sydney.",
    ],
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### Limitations

| Limitation | Impact |
|-----------|--------|
| Tool arguments are model-generated | Arguments may be malformed; validate before hitting the DB |
| No streaming | The UI waits until the full loop completes before showing the reply |
| Tool errors not surfaced to user | If a tool raises an exception the chat fails silently — add try/except in `dispatch_tool` |